# Modelo final - BETO head

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split, HahaDataset
from betohumor.models.beto import build_beto
from betohumor.train import train_model
from betohumor.metrics import get_best_macro_f1_from_history, get_training_history
from betohumor.plots import plot_training_curves

## 1. Mejor configuración (resultado del CV)
Completar con los valores reales que ganaron en `cross_validation_baseline.ipynb`.

In [ ]:
BEST_LR = 5e-4   # TODO: completar con el resultado real del CV
BEST_WD = 0.01   # TODO: completar con el resultado real del CV

## 2. Datos: train + val combinados
Se separa una porción chica (5%) solo para que el Trainer tenga con qué
medir el early stopping — no para elegir hiperparámetros, eso ya está
decidido por el CV.

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
df_combined = pd.concat([df_train, df_val]).reset_index(drop=True)

df_train_final, df_val_final = train_test_split(
    df_combined, test_size=0.05,
    stratify=df_combined[data_config.label_col],
    random_state=data_config.seed,
)

print(f'Train final: {len(df_train_final)} | Val (early stopping): {len(df_val_final)}')

## 3. Entrenar el modelo final

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

train_dataset = HahaDataset(df_train_final, tokenizer, data_config)
val_dataset   = HahaDataset(df_val_final,   tokenizer, data_config)

model = build_beto(beto_config)

trainer = train_model(
    model, train_dataset, val_dataset, beto_config,
    output_dir='results/final_beto',
    seed=data_config.seed,
    learning_rate=BEST_LR,
    weight_decay=BEST_WD,
)

## 4. Curva de entrenamiento

In [ ]:
history = get_training_history(trainer)
macro_f1 = get_best_macro_f1_from_history(trainer)

print(f'Macro F1 (early stopping val): {macro_f1:.4f}')
plot_training_curves(history)

## 5. Guardar el modelo final

In [ ]:
trainer.save_model('results/final_beto')
tokenizer.save_pretrained('results/final_beto')
print('Modelo final guardado')